In [ ]:
! pip install -q gensim

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from baseline import W2VSentenceEmbedder

df = pd.read_csv("hf://datasets/mrm8488/goemotions/goemotions.csv")

emotion_groups = {
    'contentment': ['joy', 'approval', 'caring', 'gratitude', 'love', 'pride', 'relief'],
    'awe':         ['admiration', 'realization', 'surprise'],
    'amusement':   ['amusement'],
    'excitement':  ['excitement', 'desire', 'curiosity', 'optimism'],
    'sadness':     ['sadness', 'disappointment', 'grief', 'remorse'],
    'disgust':     ['disgust', 'disapproval'],
    'fear':        ['fear', 'nervousness', 'confusion', 'embarrassment'],
    'anger':       ['anger', 'annoyance'],
}

grouped = {group: df[emotions].max(axis=1) for group, emotions in emotion_groups.items()}

all_emotions = [emotion for emotions in emotion_groups.values() for emotion in emotions]
df = df.drop(columns=all_emotions)

for group, values in grouped.items():
    df[group] = values

groups = list(emotion_groups.keys())

if 'neutral' in df.columns:
    df = df.drop(columns=['neutral'])
df = df[df[groups].sum(axis=1) > 0].reset_index(drop=True)

print(df.head())

counts = df[groups].sum().sort_values(ascending=False)

ax = counts.plot(kind="bar", color="steelblue")
ax.set_ylabel("Количество единиц")
ax.set_title("Число положительных меток по группам эмоций (multilabel)")
for i, v in enumerate(counts):
    ax.text(i, v, str(int(v)), ha="center", va="bottom")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

random_state = 42
groups = list(emotion_groups.keys())

class_counts = df[groups].sum()
print("Частоты классов до разрешения мультилейбла:")
print(class_counts.sort_values(), "\n")

M = df[groups].to_numpy()                 
freq = class_counts[groups].to_numpy()    

masked = np.where(M == 1, freq[None, :], np.iinfo(np.int64).max)
label_idx = masked.argmin(axis=1)
df["label"] = np.array(groups)[label_idx]

df_single = df[["text", "label"]].copy()
print("Распределение после разрешения мультилейбла:")
print(df_single["label"].value_counts(), "\n")

T = int(df_single["label"].value_counts().min())
print("Балансируем все классы до T =", T)

df_balanced = (
    df_single.groupby("label", group_keys=False)
             .sample(n=T, random_state=random_state)
             .sample(frac=1, random_state=random_state)   
             .reset_index(drop=True)
)

print("\nРаспределение после балансировки:")
print(df_balanced["label"].value_counts())
print("rows:", len(df_balanced))
df_balanced.head()

In [ ]:
df_balanced.to_csv("goemotions_balanced.csv", index=False)
df_balanced.to_parquet("goemotions_balanced.parquet", index=False)
print("saved:", len(df_balanced), "строк ->",
      "goemotions_balanced.csv / goemotions_balanced.parquet")
print("колонки:", list(df_balanced.columns))

In [ ]:
embedder = W2VSentenceEmbedder()

texts = df_balanced["text"].astype(str).tolist()

X = np.vstack(embedder(texts)).astype(np.float32)
y = df_balanced["label"].to_numpy()   

print("X:", X.shape, "| y:", y.shape, "| dim:", embedder.vector_size)
print("классы:", np.unique(y))

In [ ]:
! pip install -q optuna

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=100),
        "max_depth": trial.suggest_int("max_depth", 10, 50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
    }
    clf = RandomForestClassifier(
        random_state=random_state,
        n_jobs=-1,
        class_weight="balanced",
        **params,
    )
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=1)
    return scores.mean()

optuna.logging.set_verbosity(optuna.logging.INFO)

study = optuna.create_study(
    direction="maximize",
    study_name="rf_goemotions",
    storage="sqlite:///rf_goemotions.db",
    load_if_exists=True,  
)

def log_cb(study, trial):
    print(f"[trial {trial.number}] f1_macro={trial.value:.4f} | best={study.best_value:.4f}", flush=True)

study.optimize(
    objective,
    n_trials=40,
    timeout=10 * 3600,         
    callbacks=[log_cb],
    show_progress_bar=False,  
)

print("Лучший F1-macro (CV на train):", round(study.best_value, 3))
print("Лучшие параметры:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
from sklearn.metrics import classification_report

best_clf = RandomForestClassifier(
    random_state=random_state,
    n_jobs=-1,
    class_weight="balanced",
    **study.best_params,
)
best_clf.fit(X_train, y_train)
y_pred = best_clf.predict(X_test)

print("classification_report (лучшие параметры Optuna, отложенная выборка):\n")
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
import joblib

final_clf = RandomForestClassifier(
    random_state=random_state,
    n_jobs=-1,
    class_weight="balanced",
    **study.best_params,
)
final_clf.fit(X, y)

model_path = "rf_goemotions_optuna.joblib"
joblib.dump(
    {
        "model": final_clf,
        "classes": list(final_clf.classes_),
        "best_params": study.best_params,
        "best_cv_f1_macro": study.best_value,
        "embedder": "W2VSentenceEmbedder",
        "vector_size": int(X.shape[1]),
    },
    model_path,
)
print("Итоговая модель сохранена ->", model_path)

loaded = joblib.load(model_path)
print("Загружено, классы:", loaded["classes"])